## Lesson Overview

**What this lesson teaches:** the full tool-calling ("agentic") loop with more than one tool, plus a `batch_tool` pattern that lets Claude ask for several tool calls in a single turn instead of going back and forth one tool at a time.

**What's happening under the hood, step by step:**
1. **Setup cell** loads `.env` and builds the shared `client`/`MODEL_NAME`, same pattern as every other lesson.
2. **Tools are just Python functions** (`get_current_datetime`, `add_duration_to_datetime`, `set_reminder`) each paired with a JSON `input_schema` describing their arguments to the model. `set_reminder` is a local stub — it prints and returns data instead of touching a real calendar, so it's safe to run.
3. **`TOOL_REGISTRY`** maps each tool's name (string) to its actual Python function. **`dispatch_tool`** looks a name up in that registry, JSON-decodes the arguments if needed, and calls it — this indirection is what lets the loop below call *any* tool by name without an `if/elif` chain.
4. **`batch_tool`** is itself a "tool" whose implementation loops over a list of `{name, arguments}` requests and calls `dispatch_tool` on each — this is how Claude can request multiple tool calls (e.g. get the time *and* add a duration to it) in one response instead of two separate round trips.
5. **`run_tool_loop`** is the actual agent loop: send the conversation + tool schemas to Claude → if the reply has no `tool_use` blocks, you're done, return it → otherwise append the assistant's tool request to the message history, run each requested tool locally through `dispatch_tool`, package the return values as `tool_result` blocks, append those as the next user turn, and go around again (bounded by `max_rounds`).
6. The **local sanity-check cell** exercises the datetime/reminder/batch logic directly in Python with no API call, so you can confirm the tool implementations are correct before trusting the model to drive them.
7. The later worked examples show the loop actually chaining tool calls — e.g. asking for "the weekday 3 years from now" forces Claude to call `get_current_datetime` and then feed that result into `add_duration_to_datetime` before it can answer.

Compare this to Lesson 8: same loop shape, but here there's a registry of several tools and a `batch_tool` for requesting many at once, instead of one hardcoded tool.


# Lesson 7: Anthropic tools and local helpers

This notebook is split into two parts:
- local Python utilities that run on your machine without calling the API
- an optional Anthropic demo that uses `ANTHROPIC_API_KEY`

The goal is to keep the lesson runnable even when the API key is not set yet, while still showing the full tool-calling flow when credentials are available.


## Setup

Install the dependencies once in your environment:

```bash
pip install anthropic python-dotenv
```

Put your credentials in `Claude_API_Training/.env` or the project root `.env` file.
A minimal file looks like this:

```env
ANTHROPIC_API_KEY=your_key_here
MODEL_NAME=claude-haiku-4-5
```

`MODEL_NAME` is optional. If it is missing, the notebook falls back to `claude-haiku-4-5`.


In [1]:
import json
import os
from datetime import datetime, timedelta
from pathlib import Path

try:
    import anthropic
except ImportError:
    anthropic = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

def load_env_file(env_path):
    env_path = Path(env_path)
    if not env_path.exists():
        return False

    with env_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")
    return True

load_dotenv()

env_loaded = False
for candidate in (Path(".env"), Path("Claude_API_Training/.env")):
    if load_env_file(candidate):
        env_loaded = True
        break

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME", "claude-haiku-4-5")
client = anthropic.Anthropic(api_key=API_KEY) if anthropic and API_KEY else None

print(f"Environment loaded: {env_loaded}")
print(f"Model: {MODEL_NAME}")
if client is None:
    print("No Anthropic client yet. Local helpers will still run; the API demo will be skipped until ANTHROPIC_API_KEY is set.")
else:
    print("Anthropic client ready.")


Environment loaded: True
Model: claude-haiku-4-5-20251001
Anthropic client ready.


## Message helpers

These helpers keep the conversation history in the format the Anthropic Messages API expects: a list of dictionaries with `role` and `content`.

The `chat` wrapper keeps the request shape in one place so you can reuse it while experimenting with different prompts, temperatures, or tools.


In [2]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def text_from_message(message):
    return "\n".join(
        block.text for block in message.content if getattr(block, "type", None) == "text"
    )

def normalize_block(block):
    if isinstance(block, dict):
        return block
    if hasattr(block, "model_dump"):
        return block.model_dump()
    if hasattr(block, "dict"):
        return block.dict()
    return block

def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    if client is None:
        raise RuntimeError(
            "Anthropic is not ready. Install the package and set ANTHROPIC_API_KEY in .env before calling chat()."
        )

    params = {
        "model": MODEL_NAME,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences or [],
    }

    if system:
        params["system"] = system
    if tools:
        params["tools"] = tools

    return client.messages.create(**params)


## Local tools and schemas

The lesson uses a small set of time-related tools so you can see both the Python implementation and the JSON schema side by side.

`set_reminder` is a stub here. It prints and returns structured data locally so the notebook stays safe to run on your machine.


In [3]:
def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    reminder = {"content": content, "timestamp": timestamp, "status": "scheduled"}
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")
    return reminder

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "Datetime string to update. Example: 2025-04-03",
            },
            "duration": {
                "type": "number",
                "description": "Amount of time to add or subtract.",
            },
            "unit": {
                "type": "string",
                "description": "seconds, minutes, hours, days, weeks, months, or years.",
            },
            "input_format": {
                "type": "string",
                "description": "Python strptime format used to parse datetime_str.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a local reminder record and prints the reminder details.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "Reminder text.",
            },
            "timestamp": {
                "type": "string",
                "description": "When the reminder should fire.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously.",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke.",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke.",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "Tool arguments encoded as JSON.",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


In [4]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format string.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "Python strftime format string used for the output.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

def batch_tool(invocations):
    results = []
    for invocation in invocations:
        name = invocation["name"]
        arguments = invocation["arguments"]
        results.append({"name": name, "result": dispatch_tool(name, arguments)})
    return {"results": results}

TOOL_REGISTRY = {
    "get_current_datetime": get_current_datetime,
    "add_duration_to_datetime": add_duration_to_datetime,
    "set_reminder": set_reminder,
    "batch_tool": batch_tool,
}

def dispatch_tool(name, arguments):
    if isinstance(arguments, str):
        arguments = json.loads(arguments)

    if name not in TOOL_REGISTRY:
        raise ValueError(f"Unknown tool: {name}")

    return TOOL_REGISTRY[name](**arguments)

def run_tool_loop(messages, system=None, temperature=0.2, tools=None, max_rounds=5):
    if client is None:
        raise RuntimeError("Set ANTHROPIC_API_KEY before running the tool loop.")

    current_messages = list(messages)

    for _ in range(max_rounds):
        response = chat(
            current_messages,
            system=system,
            temperature=temperature,
            tools=tools,
        )

        tool_uses = [block for block in response.content if getattr(block, "type", None) == "tool_use"]
        if not tool_uses:
            return response

        current_messages.append(
            {
                "role": "assistant",
                "content": [normalize_block(block) for block in response.content],
            }
        )

        tool_results = []
        for block in tool_uses:
            result = dispatch_tool(block.name, block.input)
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                }
            )

        current_messages.append({"role": "user", "content": tool_results})

    raise RuntimeError("Tool conversation did not finish within max_rounds.")


## Local sanity checks

These cells do not need the API key. They prove the helper functions work and show the shapes of the data structures before you try a remote call.


In [5]:
demo_messages = []
add_user_message(demo_messages, "Explain how the notebook stores conversation history.")
add_assistant_message(demo_messages, "It stores each turn as a role/content dictionary.")

print("Conversation history:")
print(json.dumps(demo_messages, indent=2))

print("\nDatetime helper example:")
print(add_duration_to_datetime("2025-04-03", duration=7, unit="days"))

print("\nReminder stub example:")
print(json.dumps(set_reminder("Review Lesson 7", "2026-08-07T18:00:00"), indent=2))

print("\nBatch tool example:")
batch_example = dispatch_tool(
    "batch_tool",
    {
        "invocations": [
            {"name": "get_current_datetime", "arguments": json.dumps({"date_format": "%Y-%m-%d"})},
            {"name": "add_duration_to_datetime", "arguments": json.dumps({"datetime_str": "2025-04-03", "duration": 2, "unit": "weeks"})},
        ]
    },
)
print(json.dumps(batch_example, indent=2))


Conversation history:
[
  {
    "role": "user",
    "content": "Explain how the notebook stores conversation history."
  },
  {
    "role": "assistant",
    "content": "It stores each turn as a role/content dictionary."
  }
]

Datetime helper example:
Thursday, April 10, 2025 12:00:00 AM

Reminder stub example:
----
Setting the following reminder for 2026-08-07T18:00:00:
Review Lesson 7
----
{
  "content": "Review Lesson 7",
  "timestamp": "2026-08-07T18:00:00",
  "status": "scheduled"
}

Batch tool example:
{
  "results": [
    {
      "name": "get_current_datetime",
      "result": "2026-08-07"
    },
    {
      "name": "add_duration_to_datetime",
      "result": "Thursday, April 17, 2025 12:00:00 AM"
    }
  ]
}


## Optional API demo

Run this only after `ANTHROPIC_API_KEY` is available. The cell sends a short prompt with the tool schemas attached so the model can decide whether to call a tool.

If the key is missing, the cell prints a skip message instead of failing the notebook.


In [6]:
if client is None:
    print("Skipping API demo. Add ANTHROPIC_API_KEY to .env and rerun this cell.")
else:
    messages = []
    add_user_message(messages, "What is the current date and time? Use a tool if it helps.")

    response = run_tool_loop(
        messages,
        system="You are a concise assistant. Use tools when they are helpful and return a brief answer.",
        tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
            batch_tool_schema,
        ],
    )

    print(text_from_message(response))


The current date and time is **August 7, 2026 at 2:17:57 PM** (14:17:57).


## Summary: how to use this notebook

1. Open the notebook from `Claude_API_Training` in the same environment where your `.env` file lives.
2. Run the setup cell first. It loads environment variables, creates the Anthropic client when a key is present, and leaves the local helpers available either way.
3. Run the local sanity-check cells to verify the datetime helpers, reminder stub, and batch tool logic.
4. Add `ANTHROPIC_API_KEY` to `.env` if you want the remote API demo to run.
5. Run the optional API demo cell to see the assistant call tools and print the final response.
6. Extend `TOOL_REGISTRY` and the schema cells when you want to add more tools or adapt the notebook to a new lesson.

The notebook is intentionally structured so the non-API pieces still execute on your machine even before credentials are configured.


In [7]:
# Example: add 100 days to the current date and time through the tool registry
current_timestamp = get_current_datetime()

future_timestamp = dispatch_tool(
    "add_duration_to_datetime",
    {
        "datetime_str": current_timestamp,
        "duration": 100,
        "unit": "days",
        "input_format": "%Y-%m-%d %H:%M:%S",
    },
)

print("Current timestamp:", current_timestamp)
print("100 days later:", future_timestamp)


Current timestamp: 2026-08-07 14:17:59
100 days later: Sunday, November 15, 2026 02:17:59 PM


In [10]:
# Example: Claude using tools to get the right answer

messages = []
add_user_message(
    messages,
    "What is the current date and time, and what will by the week day in 3 years?"
)

response = run_tool_loop(
    messages,
    system="You are a concise assistant. Use tools when they are helpful and return a brief answer.",
    tools=[
        get_current_datetime_schema,
        add_duration_to_datetime_schema,
        set_reminder_schema,
        batch_tool_schema,
    ],
)

print(text_from_message(response))

**Current date and time:** Friday, August 7, 2026 at 2:27:58 PM

**In 3 years:** Monday, August 7, 2029
